In [1]:
#!/usr/bin/env python3
"""
aggregate_results.py

Amasses all CellReg comparison results into a single JSON:
  - all_results_{nc}n.json                       (original, no perturbation)
  - all_results_{nc}n_perturbed_{PRESET}.json    (perturbed)
      • for nc=1000, perturbed results are split across:
          CellReg_1000_batch_perturbed_{PRESET}            (Tier A — legacy folder)
          CellReg_1000n_Tier_B_perturbed_{PRESET}          (Tier B)
          CellReg_1000n_Tier_C_perturbed_{PRESET}          (Tier C)
        these are merged into a single 1000n entry.
  - matching_results.npy                         (temporal correlation test)

Output: combined_results.json in BASE_DIR
"""

import json
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison")
NEURON_COUNTS = [100, 250, 500, 1000]
PRESET = "moderate"   # change to whichever preset you ran
OUT_PATH = BASE_DIR / "combined_results.json"


def to_jsonable(obj):
    """Recursively convert numpy types/arrays to JSON-friendly Python types."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def load_json(p):
    if not p.exists():
        print(f"  MISSING: {p}")
        return None
    with open(p) as f:
        return json.load(f)


def main():
    combined = {
        "benchmarks": {
            "original":            {},   # nc -> contents of all_results_{nc}n.json
            f"perturbed_{PRESET}": {},
        },
        "temporal_correlation": None,
    }

    # Benchmark JSONs
    for nc in NEURON_COUNTS:
        # ── original (no perturbation) ──
        orig = BASE_DIR / f"CellReg_{nc}_batch" / f"all_results_{nc}n.json"
        d_orig = load_json(orig)
        if d_orig is not None:
            combined["benchmarks"]["original"][str(nc)] = d_orig
            print(f"  loaded original {nc}n")

        # ── perturbed ──
        if nc == 1000:
            # 1000n is split: Tier A in the legacy folder, B & C in tier-suffixed folders
            merged = {}
            legacy = BASE_DIR / f"CellReg_{nc}_batch_perturbed_{PRESET}" / f"all_results_{nc}n_perturbed_{PRESET}.json"
            d_a = load_json(legacy)
            if d_a is not None:
                merged.update(d_a)
                print(f"  loaded perturbed 1000n Tier A (legacy folder)")
            for tier in ("B", "C"):
                p = BASE_DIR / f"CellReg_{nc}n_Tier_{tier}_perturbed_{PRESET}" / f"all_results_{nc}n_perturbed_{PRESET}.json"
                d = load_json(p)
                if d is not None:
                    merged.update(d)
                    print(f"  loaded perturbed 1000n Tier {tier}")
            if merged:
                combined["benchmarks"][f"perturbed_{PRESET}"][str(nc)] = merged
        else:
            pert = BASE_DIR / f"CellReg_{nc}_batch_perturbed_{PRESET}" / f"all_results_{nc}n_perturbed_{PRESET}.json"
            d_pert = load_json(pert)
            if d_pert is not None:
                combined["benchmarks"][f"perturbed_{PRESET}"][str(nc)] = d_pert
                print(f"  loaded perturbed {nc}n")

    # Temporal correlation .npy
    tc_path = BASE_DIR / "temporal_correlation_test" / "matching_results.npy"
    if tc_path.exists():
        tc = np.load(tc_path, allow_pickle=True).item()
        combined["temporal_correlation"] = to_jsonable(tc)
        print(f"  loaded temporal_correlation ({len(tc)} durations)")
    else:
        print(f"  MISSING: {tc_path}")

    # Write
    with open(OUT_PATH, "w") as f:
        json.dump(combined, f, indent=2)

    size_mb = OUT_PATH.stat().st_size / 1e6
    print(f"\n  → {OUT_PATH}  ({size_mb:.2f} MB)")


if __name__ == "__main__":
    main()


  loaded original 100n
  loaded perturbed 100n
  loaded original 250n
  loaded perturbed 250n
  loaded original 500n
  loaded perturbed 500n
  loaded original 1000n
  loaded perturbed 1000n Tier A (legacy folder)
  loaded perturbed 1000n Tier B
  loaded perturbed 1000n Tier C
  loaded temporal_correlation (4 durations)

  → C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison\combined_results.json  (11.26 MB)


In [4]:
#!/usr/bin/env python3
"""
analyze_results.py

Nature-figure-grade stats on combined_results.json:
  - Per-condition paired comparisons (Stars2Cells vs CellReg) across
    the 40 runs/condition (8 animals x 5 seeds)
  - Wilcoxon signed-rank (paired, two-sided)
  - Cohen's d_z (paired effect size)
  - Bootstrap 95% CI on the mean paired difference
  - Win rate (% runs where S2C >= CellReg)
  - Holm-Bonferroni correction across 8 conditions, per neuron count
  - Pooled meta comparisons across all 1280 paired runs
  - Temporal correlation vs chance (one-sample Wilcoxon)

Outputs:
  stats_summary.json   - structured numbers for the figure
  stats_report.txt     - human-readable summary
"""

import json
import numpy as np
from pathlib import Path
from scipy import stats

# ──────────────────────────────────────────────────────────────────────────
BASE_DIR = Path(r"C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison")
IN_PATH  = BASE_DIR / "combined_results.json"
OUT_JSON = BASE_DIR / "stats_summary.json"
OUT_TXT  = BASE_DIR / "stats_report.txt"

NEURON_COUNTS = [100, 250, 500, 1000]
CONDITIONS = ["A_rot", "A_trans", "A_combined",
              "B_dropout", "B_drift", "B_rot", "B_combined",
              "C_walk"]
METRICS = ["f1", "precision", "recall"]
N_BOOT = 10_000
RNG = np.random.default_rng(0)


# ──────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────

def cohens_dz(diff):
    """Paired Cohen's d_z = mean(diff) / std(diff)."""
    diff = np.asarray(diff, dtype=float)
    if diff.size < 2 or np.std(diff, ddof=1) == 0:
        return np.nan
    return float(np.mean(diff) / np.std(diff, ddof=1))


def boot_ci(x, n_boot=N_BOOT, alpha=0.05, statistic=np.mean):
    """Percentile bootstrap CI."""
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return (np.nan, np.nan)
    idx = RNG.integers(0, x.size, size=(n_boot, x.size))
    boots = statistic(x[idx], axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def holm_bonferroni(pvals):
    """Return Holm-corrected p-values, preserving input order."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    corrected = np.empty(n, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = p[idx] * (n - rank)
        running_max = max(running_max, adj)
        corrected[idx] = min(running_max, 1.0)
    return corrected.tolist()


def stars(p):
    if np.isnan(p): return "n/a"
    if p < 1e-4:    return "****"
    if p < 1e-3:    return "***"
    if p < 1e-2:    return "**"
    if p < 5e-2:    return "*"
    return "ns"


def fmt_pct(x):
    return f"{x*100:5.2f}%" if not np.isnan(x) else "    —"


# ──────────────────────────────────────────────────────────────────────────
# Build paired arrays from per-run results
# ──────────────────────────────────────────────────────────────────────────

def collect_paired(block, condition, metric):
    """
    Return (s2c, cr) arrays of length n_runs for a given condition+metric.
    Uses results[*]['stars2cells'|'cellreg']['aggregate'][metric].
    Aligned by run-key (animal_condition_seed).
    """
    s2c, cr = [], []
    for k, r in block["results"].items():
        if r.get("condition") != condition:
            continue
        s = r.get("stars2cells", {}).get("aggregate", {}).get(metric)
        c = r.get("cellreg", {}).get("aggregate", {}).get(metric)
        if s is None or c is None:
            continue
        s2c.append(float(s))
        cr.append(float(c))
    return np.array(s2c), np.array(cr)


def collect_paired_pooled(block, metric):
    """Pool across all conditions for an overall comparison."""
    s2c_all, cr_all = [], []
    for cond in CONDITIONS:
        s, c = collect_paired(block, cond, metric)
        s2c_all.append(s); cr_all.append(c)
    return np.concatenate(s2c_all), np.concatenate(cr_all)


# ──────────────────────────────────────────────────────────────────────────
# Comparison block
# ──────────────────────────────────────────────────────────────────────────

def compare(s2c, cr):
    """Run paired stats on two aligned arrays."""
    n = len(s2c)
    out = {"n": int(n)}
    if n < 2:
        return out

    diff = s2c - cr
    out["s2c_mean"]   = float(np.mean(s2c))
    out["s2c_sem"]    = float(stats.sem(s2c)) if n > 1 else float("nan")
    out["s2c_median"] = float(np.median(s2c))
    out["cr_mean"]    = float(np.mean(cr))
    out["cr_sem"]     = float(stats.sem(cr)) if n > 1 else float("nan")
    out["cr_median"]  = float(np.median(cr))

    out["mean_diff"]  = float(np.mean(diff))
    out["median_diff"] = float(np.median(diff))
    out["sem_diff"]   = float(stats.sem(diff)) if n > 1 else float("nan")

    lo, hi = boot_ci(diff)
    out["diff_ci95"] = [lo, hi]

    out["cohens_dz"] = cohens_dz(diff)
    out["win_rate_s2c"] = float(np.mean(diff > 0))
    out["tie_rate"]     = float(np.mean(diff == 0))

    # Wilcoxon signed-rank, two-sided. Skip if all zero diffs.
    if np.all(diff == 0):
        out["wilcoxon_W"] = float("nan")
        out["wilcoxon_p"] = 1.0
    else:
        try:
            w = stats.wilcoxon(s2c, cr, zero_method="wilcox",
                               alternative="two-sided")
            out["wilcoxon_W"] = float(w.statistic)
            out["wilcoxon_p"] = float(w.pvalue)
        except ValueError:
            out["wilcoxon_W"] = float("nan")
            out["wilcoxon_p"] = float("nan")

    # Paired t-test as supplement (some reviewers like it)
    try:
        t = stats.ttest_rel(s2c, cr)
        out["ttest_t"] = float(t.statistic)
        out["ttest_p"] = float(t.pvalue)
    except Exception:
        out["ttest_t"] = float("nan")
        out["ttest_p"] = float("nan")

    return out


# ──────────────────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────────────────

def main():
    with open(IN_PATH) as f:
        data = json.load(f)

    summary = {"by_neuron_count": {}, "pooled": {}, "temporal_correlation": {}}
    txt_lines = []
    P = txt_lines.append

    P("=" * 78)
    P("  Stars2Cells vs CellReg — paired statistical comparison")
    P("  (8 animals × 5 seeds = 40 paired runs per condition)")
    P("=" * 78)

    for state in ["original", "perturbed_moderate"]:
        P(f"\n{'#' * 78}")
        P(f"#  {state.upper()}")
        P(f"{'#' * 78}")

        for nc in NEURON_COUNTS:
            block = data["benchmarks"][state].get(str(nc))
            if block is None:
                continue

            key = f"{state}_{nc}n"
            summary["by_neuron_count"][key] = {"per_condition": {}}

            P(f"\n┌{'─' * 76}┐")
            P(f"│  {nc} neurons — {state}")
            P(f"└{'─' * 76}┘")

            # ── Per condition ────────────────────────────────────────────
            for metric in METRICS:
                pvals_cond = []
                rows = []
                for cond in CONDITIONS:
                    s, c = collect_paired(block, cond, metric)
                    cmp = compare(s, c)
                    summary["by_neuron_count"][key]["per_condition"].setdefault(cond, {})[metric] = cmp
                    pvals_cond.append(cmp.get("wilcoxon_p", np.nan))
                    rows.append((cond, cmp))

                # Holm correction across 8 conditions
                p_holm = holm_bonferroni([np.nan_to_num(p, nan=1.0) for p in pvals_cond])
                for (cond, cmp), ph in zip(rows, p_holm):
                    cmp["wilcoxon_p_holm"] = float(ph)

                P(f"\n  ── {metric.upper()} ──")
                P(f"  {'condition':<12}  {'S2C mean±SEM':>14}  {'CR mean±SEM':>14}  "
                  f"{'Δ (S2C−CR)':>12}  {'95% CI':>20}  {'d_z':>6}  "
                  f"{'win%':>5}  {'p_Holm':>9}  sig")
                P(f"  {'-'*12}  {'-'*14}  {'-'*14}  {'-'*12}  {'-'*20}  {'-'*6}  "
                  f"{'-'*5}  {'-'*9}  ---")
                for cond, cmp in rows:
                    if cmp.get("n", 0) < 2:
                        P(f"  {cond:<12}  (insufficient data)")
                        continue
                    ci = cmp["diff_ci95"]
                    P(f"  {cond:<12}  "
                      f"{cmp['s2c_mean']*100:5.1f}±{cmp['s2c_sem']*100:4.1f}%   "
                      f"{cmp['cr_mean']*100:5.1f}±{cmp['cr_sem']*100:4.1f}%   "
                      f"{cmp['mean_diff']*100:+6.1f} pp   "
                      f"[{ci[0]*100:+5.1f}, {ci[1]*100:+5.1f}] pp   "
                      f"{cmp['cohens_dz']:+5.2f}  "
                      f"{cmp['win_rate_s2c']*100:4.0f}%  "
                      f"{cmp['wilcoxon_p_holm']:.2e}  {stars(cmp['wilcoxon_p_holm'])}")

            # ── Overall (pooled across conditions for this nc) ───────────
            P(f"\n  ── OVERALL (pooled across all 8 conditions, n=320 paired runs) ──")
            summary["by_neuron_count"][key]["overall"] = {}
            for metric in METRICS:
                s, c = collect_paired_pooled(block, metric)
                cmp = compare(s, c)
                summary["by_neuron_count"][key]["overall"][metric] = cmp
                ci = cmp["diff_ci95"]
                P(f"  {metric:<10}  "
                  f"S2C={cmp['s2c_mean']*100:5.1f}±{cmp['s2c_sem']*100:4.1f}%   "
                  f"CR={cmp['cr_mean']*100:5.1f}±{cmp['cr_sem']*100:4.1f}%   "
                  f"Δ={cmp['mean_diff']*100:+5.1f}pp "
                  f"[{ci[0]*100:+5.1f},{ci[1]*100:+5.1f}]   "
                  f"d_z={cmp['cohens_dz']:+5.2f}   "
                  f"win={cmp['win_rate_s2c']*100:.0f}%   "
                  f"p={cmp['wilcoxon_p']:.2e} {stars(cmp['wilcoxon_p'])}")

    # ── Grand pooled (all neuron counts, all conditions) ──
    P(f"\n{'#' * 78}")
    P("#  GRAND POOLED — every paired run across every condition & neuron count")
    P(f"{'#' * 78}")
    for state in ["original", "perturbed_moderate"]:
        summary["pooled"][state] = {}
        s_all, c_all = {m: [] for m in METRICS}, {m: [] for m in METRICS}
        for nc in NEURON_COUNTS:
            block = data["benchmarks"][state].get(str(nc))
            if block is None:
                continue
            for metric in METRICS:
                s, c = collect_paired_pooled(block, metric)
                s_all[metric].append(s); c_all[metric].append(c)
        P(f"\n  {state}:")
        for metric in METRICS:
            s = np.concatenate(s_all[metric]); c = np.concatenate(c_all[metric])
            cmp = compare(s, c)
            summary["pooled"][state][metric] = cmp
            ci = cmp["diff_ci95"]
            P(f"    {metric:<10}  n={cmp['n']:4d}   "
              f"S2C={cmp['s2c_mean']*100:5.1f}%   CR={cmp['cr_mean']*100:5.1f}%   "
              f"Δ={cmp['mean_diff']*100:+5.1f}pp [{ci[0]*100:+5.1f},{ci[1]*100:+5.1f}]   "
              f"d_z={cmp['cohens_dz']:+5.2f}   "
              f"win={cmp['win_rate_s2c']*100:.0f}%   "
              f"p={cmp['wilcoxon_p']:.2e} {stars(cmp['wilcoxon_p'])}")

    # ── Temporal correlation vs chance ──
    P(f"\n{'#' * 78}")
    P("#  TEMPORAL CORRELATION CONTROL (1000 neurons, identical positions)")
    P(f"{'#' * 78}")
    tc = data.get("temporal_correlation") or {}
    n_neurons_tc = 1000
    chance_pct = 100.0 / n_neurons_tc
    P(f"\n  Chance level = 1/{n_neurons_tc} = {chance_pct:.3f}% per neuron")
    P(f"  {'duration':>10}  {'n_pairs':>8}  {'mean acc':>10}  {'95% CI':>22}  "
      f"{'p (vs chance)':>15}")
    for dur in sorted(tc.keys(), key=lambda x: int(x)):
        rec = tc[dur]
        per_pair_acc = np.array(rec["pair_correct"]) / n_neurons_tc * 100
        ci = boot_ci(per_pair_acc)
        try:
            w = stats.wilcoxon(per_pair_acc - chance_pct,
                               alternative="two-sided",
                               zero_method="wilcox")
            p = float(w.pvalue)
        except ValueError:
            p = float("nan")
        summary["temporal_correlation"][dur] = {
            "n_pairs": len(per_pair_acc),
            "mean_accuracy_pct": float(np.mean(per_pair_acc)),
            "median_accuracy_pct": float(np.median(per_pair_acc)),
            "ci95_pct": [ci[0], ci[1]],
            "chance_pct": chance_pct,
            "wilcoxon_vs_chance_p": p,
        }
        P(f"  {dur:>7} min  {len(per_pair_acc):>8d}  "
          f"{np.mean(per_pair_acc):7.3f}%   "
          f"[{ci[0]:5.3f}, {ci[1]:5.3f}]%   "
          f"{p:.3f} {stars(p)}")
    P(f"\n  → Even at 180 min (108,000 frames), accuracy ≈ chance.")
    P(f"    Temporal correlation cannot register cells without spatial info.")

    # ── Headline numbers (for figure caption) ──
    P(f"\n{'=' * 78}")
    P("  HEADLINE NUMBERS FOR FIGURE / CAPTION")
    P(f"{'=' * 78}")
    for state in ["original", "perturbed_moderate"]:
        for nc in NEURON_COUNTS:
            key = f"{state}_{nc}n"
            blk = summary["by_neuron_count"].get(key, {}).get("overall", {}).get("f1")
            if not blk:
                continue
            ci = blk["diff_ci95"]
            P(f"  {state:<22} {nc:>5}n  →  "
              f"Δ_F1 = {blk['mean_diff']*100:+5.1f} pp "
              f"[{ci[0]*100:+5.1f}, {ci[1]*100:+5.1f}], "
              f"d_z={blk['cohens_dz']:+.2f}, "
              f"win={blk['win_rate_s2c']*100:.0f}%, "
              f"p={blk['wilcoxon_p']:.1e}")

    # Write outputs
    OUT_TXT.write_text("\n".join(txt_lines))
    with open(OUT_JSON, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n".join(txt_lines))
    print(f"\n  → Wrote {OUT_JSON}")
    print(f"  → Wrote {OUT_TXT}")


if __name__ == "__main__":
    main()

  Stars2Cells vs CellReg — paired statistical comparison
  (8 animals × 5 seeds = 40 paired runs per condition)

##############################################################################
#  ORIGINAL
##############################################################################

┌────────────────────────────────────────────────────────────────────────────┐
│  100 neurons — original
└────────────────────────────────────────────────────────────────────────────┘

  ── F1 ──
  condition       S2C mean±SEM     CR mean±SEM    Δ (S2C−CR)                95% CI     d_z   win%     p_Holm  sig
  ------------  --------------  --------------  ------------  --------------------  ------  -----  ---------  ---
  A_rot          99.6± 0.1%    76.1± 4.5%    +23.5 pp   [+15.2, +32.5] pp   +0.82    88%  2.33e-09  ****
  A_trans        99.4± 0.1%    97.6± 0.5%     +1.8 pp   [ +0.8,  +2.8] pp   +0.53    78%  7.14e-04  ***
  A_combined     99.4± 0.1%    68.9± 4.3%    +30.5 pp   [+22.4, +38.9] pp   +1.12  

In [3]:
# ──────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC + FIX: 1000n merge bug for BOTH regimes in aggregate_results.py
# ──────────────────────────────────────────────────────────────────────────
# The original aggregate cell has TWO problems for nc=1000:
#
#   1) In the perturbed branch, merged.update(d_tier) is called at the TOP
#      level of each Tier's JSON. Each Tier file has its own "results" sub-
#      dict, so every update REPLACES the prior Tier's runs:
#        step 1: merged.update(tier_A)  → merged["results"] = A's runs
#        step 2: merged.update(tier_B)  → merged["results"] = B's runs (A lost)
#        step 3: merged.update(tier_C)  → merged["results"] = C's runs (B lost)
#
#   2) In the original (best-case) branch, there is NO Tier-B/C merge at all:
#        orig = BASE_DIR / f"CellReg_{nc}_batch" / f"all_results_{nc}n.json"
#        d_orig = load_json(orig)
#      The script never even looks at `CellReg_1000n_Tier_B/` or
#      `CellReg_1000n_Tier_C/` for the original regime. So Tier-B/C runs at
#      1000n in the best-case regime exist on disk but are silently invisible
#      to the analysis. That's why the best-case 1000n analysis only has
#      Tier-A in the stats output and the productive-cell count is 27 instead
#      of 32.
#
# This cell deep-merges Tier A + B + C for nc=1000 in BOTH regimes and
# patches combined_results.json in place. Re-run analyze_results.py
# afterward.
#
# Permanent fix at the source (aggregate_results.py): apply the same
# 3-tier merge logic to the original branch when nc==1000, and replace
# the top-level merged.update(d) calls with a deep merge of the
# "results" sub-dict.
#
# Folder convention assumed (adjust paths if yours differ):
#   Best-case (original):
#     CellReg_1000_batch              / all_results_1000n.json   (Tier A)
#     CellReg_1000n_Tier_B            / all_results_1000n.json   (Tier B)
#     CellReg_1000n_Tier_C            / all_results_1000n.json   (Tier C)
#   Realistic (perturbed_moderate):
#     CellReg_1000_batch_perturbed_moderate    / all_results_1000n_perturbed_moderate.json   (Tier A)
#     CellReg_1000n_Tier_B_perturbed_moderate  / all_results_1000n_perturbed_moderate.json   (Tier B)
#     CellReg_1000n_Tier_C_perturbed_moderate  / all_results_1000n_perturbed_moderate.json   (Tier C)
# ──────────────────────────────────────────────────────────────────────────

import json
from pathlib import Path

BASE_DIR = Path(r"C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison")
PRESET   = "moderate"
nc       = 1000


def deep_merge_tiers(tier_paths, label):
    """Merge per-tier JSONs by concatenating their `results` sub-dicts."""
    merged_block = None
    per_tier_n   = {}
    print(f"\n[{label}]")
    for tier, p in tier_paths.items():
        if not p.exists():
            print(f"  MISSING Tier {tier}: {p}")
            continue
        with open(p) as f:
            d = json.load(f)
        n = len(d.get("results", {}))
        per_tier_n[tier] = n
        if merged_block is None:
            merged_block = {k: v for k, v in d.items() if k != "results"}
            merged_block["results"] = dict(d.get("results", {}))
            print(f"  Tier {tier}: {n} runs  (seeded merged block)")
        else:
            prior = set(merged_block["results"])
            new   = d.get("results", {})
            coll  = prior & set(new)
            merged_block["results"].update(new)
            msg = f"  Tier {tier}: +{n} runs  (total now {len(merged_block['results'])})"
            if coll:
                msg += f"   ⚠ {len(coll)} run-key collisions with earlier tiers"
            print(msg)
    return merged_block, per_tier_n


# ── Realistic (perturbed_moderate) ──
realistic_paths = {
    "A": BASE_DIR / f"CellReg_{nc}_batch_perturbed_{PRESET}"   / f"all_results_{nc}n_perturbed_{PRESET}.json",
    "B": BASE_DIR / f"CellReg_{nc}n_Tier_B_perturbed_{PRESET}" / f"all_results_{nc}n_perturbed_{PRESET}.json",
    "C": BASE_DIR / f"CellReg_{nc}n_Tier_C_perturbed_{PRESET}" / f"all_results_{nc}n_perturbed_{PRESET}.json",
}
realistic_block, realistic_n = deep_merge_tiers(realistic_paths, "realistic / perturbed_moderate")

# ── Best-case (original) ──
bestcase_paths = {
    "A": BASE_DIR / f"CellReg_{nc}_batch"        / f"all_results_{nc}n.json",
    "B": BASE_DIR / f"CellReg_{nc}n_Tier_B"      / f"all_results_{nc}n.json",
    "C": BASE_DIR / f"CellReg_{nc}n_Tier_C"      / f"all_results_{nc}n.json",
}
bestcase_block, bestcase_n = deep_merge_tiers(bestcase_paths, "best-case / original")

# ── Patch combined_results.json ──
combined_path = BASE_DIR / "combined_results.json"
with open(combined_path) as f:
    combined = json.load(f)

if realistic_block is not None:
    combined["benchmarks"][f"perturbed_{PRESET}"][str(nc)] = realistic_block
if bestcase_block is not None:
    combined["benchmarks"]["original"][str(nc)] = bestcase_block

with open(combined_path, "w") as f:
    json.dump(combined, f, indent=2)

print(f"\n  → patched {combined_path.name}")
if realistic_block is not None:
    print(f"  → 1000n perturbed_{PRESET}: {len(realistic_block['results'])} runs across all tiers")
if bestcase_block is not None:
    print(f"  → 1000n original (best-case): {len(bestcase_block['results'])} runs across all tiers")
print(f"\n  Re-run analyze_results.py to refresh both grand-pooled stats.")
print(f"  After re-run, send the new output and I'll fill in:")
print(f"    • best-case grand-pooled n, ΔF1, 95% CI, d_z, dropped-run count")
print(f"    • best-case 1000n pooled and per-condition F1 if relevant")


[realistic / perturbed_moderate]
  Tier A: 120 runs  (seeded merged block)
  Tier B: +160 runs  (total now 280)
  Tier C: +40 runs  (total now 320)

[best-case / original]
  Tier A: 120 runs  (seeded merged block)
  MISSING Tier B: C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison\CellReg_1000n_Tier_B\all_results_1000n.json
  MISSING Tier C: C:\Users\ariAccount\Desktop\Stars2CellsPaper\CellRegComparison\CellReg_1000n_Tier_C\all_results_1000n.json

  → patched combined_results.json
  → 1000n perturbed_moderate: 320 runs across all tiers
  → 1000n original (best-case): 120 runs across all tiers

  Re-run analyze_results.py to refresh both grand-pooled stats.
  After re-run, send the new output and I'll fill in:
    • best-case grand-pooled n, ΔF1, 95% CI, d_z, dropped-run count
    • best-case 1000n pooled and per-condition F1 if relevant
